In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")

from utils.scaling import fit_scalers, apply_scaler
from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri75anno_articulatory_features

In [2]:
df = pd.read_csv("../../data_processing/dataset_construction/datasets/processed_aemri75anno.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])
token_df = dataset_df_to_token_df(df)

In [3]:
# token_df = token_df.sample(frac=1).reset_index(drop=True)
# token_df = token_df.iloc[int(0.95 * len(token_df)):]
# token_df.sort_values(by='start', inplace=True)
# token_df.sort_values(by='ac_f', inplace=True)

In [4]:
with open(f"../../encoder_training/scalers/aemri75anno_art.pkl", "rb") as f:
    art_scalers = pickle.load(f)
with open(f"../../encoder_training/scalers/aemri75anno_ac.pkl", "rb") as f:
    ac_scalers = pickle.load(f)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import SequenceAutoencoder

device = "mps" if torch.backends.mps.is_available() else "cpu"

art_model = SequenceAutoencoder(
    feature_shape=(174, 2),
    latent_dim=64,
).to(device)
ckpt = torch.load(
    "../../encoder_training/ckpts/aemri75anno/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/aemri75anno/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [6]:
art_vectors = []
ac_vectors = []
phonemes = []
subjects = []
groups = []

pbar = tqdm.tqdm(token_df.iterrows(), total=len(token_df))

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]

    articulatory_features = get_aemri75anno_articulatory_features(art_f, start, end)
    if articulatory_features is None: continue
    articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 174, 2)
    articulatory_features = apply_scaler(articulatory_features, art_scalers[subject])
    articulatory_features = torch.from_numpy(articulatory_features).float().unsqueeze(0).to(device)
    art_out, art_z = art_model(articulatory_features, return_latent=True)

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is None: continue
    mfccs = apply_scaler(mfccs, ac_scalers[subject])
    mfccs = torch.from_numpy(mfccs).unsqueeze(0).to(device)
    ac_out, ac_z = ac_model(mfccs, return_latent=True)

    art_vectors.append(art_z.detach().cpu().numpy().squeeze())
    ac_vectors.append(ac_z.detach().cpu().numpy().squeeze())
    phonemes.append(phone)
    subjects.append(subject)
    groups.append(art_f)

100%|██████████| 37849/37849 [1:59:49<00:00,  5.26it/s]  


In [7]:
import numpy as np
art = np.stack(art_vectors, axis=0)   # (N, T_A, D_A)
ac = np.stack(ac_vectors, axis=0)  # (N, T_X, D_X)

# A: (N, T_A, D_A)
# X: (N, T_X, D_X)
N, D = art.shape
N2, D_X = ac.shape
assert N == N2

art_flat = art.reshape(N, D)
ac_flat = ac.reshape(N, D_X)   # shape (N, x_dim)

print("art_flat:", art_flat.shape, "ac_flat:", ac_flat.shape)

art_flat: (36236, 64) ac_flat: (36236, 64)


In [8]:
art_raw = art_flat.astype(np.float32)
ac_raw = ac_flat.astype(np.float32)

In [9]:
from utils import mi

res = mi.compute_all(art_raw, ac_raw, phonemes, groups)
df  = mi.per_phoneme(res)

N=36236  art=64d  ac=64d  phones=43  device=mps

=== art|ac,phone ===
  [art|ac,phone] fold 0: val NLL    53.75   sigmas at floor  0.0%
  [art|ac,phone] fold 1: val NLL    55.16   sigmas at floor  0.0%
  [art|ac,phone] fold 2: val NLL    67.18   sigmas at floor  0.0%
  [art|ac,phone] fold 3: val NLL    58.80   sigmas at floor  0.0%
  [art|ac,phone] fold 4: val NLL    55.66   sigmas at floor  0.0%

=== art|phone ===
  [art|phone] fold 0: val NLL    59.62   sigmas at floor  0.0%
  [art|phone] fold 1: val NLL    58.98   sigmas at floor  0.0%
  [art|phone] fold 2: val NLL    72.07   sigmas at floor  0.0%
  [art|phone] fold 3: val NLL    60.80   sigmas at floor  0.0%
  [art|phone] fold 4: val NLL    57.11   sigmas at floor  0.0%

=== art|ac ===
  [art|ac] fold 0: val NLL    52.62   sigmas at floor  0.0%
  [art|ac] fold 1: val NLL    55.55   sigmas at floor  0.0%
  [art|ac] fold 2: val NLL    66.76   sigmas at floor  0.0%
  [art|ac] fold 3: val NLL    57.13   sigmas at floor  0.0%
  [art|ac]

In [11]:
df.to_csv("res/aemri75anno.csv", index=False)